3 families of transformers bert tf and gpt

In [1]:
from transformers import pipeline

# mask is  just  a temporary vector which is altered base dn its previous or upcoming words and u get teh output 

fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

result = fill_mask(
    "The cat sat on the [MASK]."
)

print(result)

c:\Users\ASUS\PycharmProjects\Machine-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2977.91it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.31445756554603577, 'token': 2723, 'token_str': 'floor', 'sequence': 'the cat sat on the floor.'}, {'score': 0.11902374029159546, 'token': 2793, 'token_str': 'bed', 'sequence': 'the cat sat on the bed.'}, {'score': 0.10695982724428177, 'token': 6411, 'token_str': 'couch', 'sequence': 'the cat sat on the couch.'}, {'score': 0.06032617762684822, 'token': 10682, 'token_str': 'sofa', 'sequence': 'the cat sat on the sofa.'}, {'score': 0.05508220195770264, 'token': 2598, 'token_str': 'ground', 'sequence': 'the cat sat on the ground.'}]


# a sentence is converted in this format "i love You" -> "[CLS] i love you [SEP] CLS contains the summary of the sentence 

# Whole sentence
#     ↓
# Compressed summary
#     ↓
# [CLS]

# 3 famileies are bert -> bidirectional encoded representations of transformers -> understant context and meaning of words -> encoder
# Gpt - genrative pre trained transformer ->  predict next word -> decoder
# t5 - encoder and decoder -> does translation, corrects grammer, summarizer and all

# BERT = Reader → Reads the whole sentence → Encoder-only → Masked Language Modeling → Best for understanding (Sentiment Analysis, NER, QA).
# GPT = Writer → Predicts the next token → Decoder-only → Best for generation (chat, code, stories).
# T5 = Translator → Converts one piece of text into another → Encoder + Decoder → Best for translation, summarization, grammar correction, and text-to-text tasks.

# Model	                                                    Why
# BERT	                    Sees both left and right context, making it excellent at understanding text.
# GPT	                    Only sees previous tokens, enabling natural autoregressive generation.
# T5	                    Separates understanding (encoder) from generation (decoder), making it ideal for transforming one piece of text into another.

In [1]:
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM
)
import evaluate
import numpy as np
import pandas as pd


c:\Users\ASUS\PycharmProjects\Machine-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("stanfordnlp/imdb")
dataset.shape

# 0 → Negative
# 1 → Positive

{'train': (25000, 2), 'test': (25000, 2), 'unsupervised': (50000, 2)}

In [3]:
dataset['train'][2]

{'text': "If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />",
 'label': 0}

In [4]:
small_train = dataset["train"].shuffle(seed=42).select(range(3000))

small_test = dataset["test"].shuffle(seed=42).select(range(1000))

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length"
    )

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3752.72it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
train_dataset = small_train.map(tokenize, batched=True)

test_dataset = small_test.map(tokenize, batched=True)

In [8]:
accuracy = evaluate.load("accuracy")

In [9]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )
    


In [13]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=4,
    
    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=50
)

In [14]:
# substitution of the for loop -> load model loss calulation back propogation optimizer etc
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

trainer.train(resume_from_checkpoint='results/checkpoint-375')

Epoch,Training Loss,Validation Loss,Accuracy
2,0.293345,0.374501,0.890000
3,0.171743,0.510407,0.878000
4,0.057524,0.555426,0.884000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]


TrainOutput(global_step=1500, training_loss=0.11197180978457133, metrics={'train_runtime': 527.1477, 'train_samples_per_second': 22.764, 'train_steps_per_second': 2.846, 'total_flos': 1589608783872000.0, 'train_loss': 0.11197180978457133, 'epoch': 4.0})

In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 has no padding token by default
tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10571.65it/s]


# Temperature: Controls the randomness of the output. Lower = more predictable, Higher = more creative.
# max_new_tokens: Sets the maximum number of new tokens the model can generate. Lower = shorter output, Higher = longer output.
# top_p: Limits the pool of candidate tokens. Lower = safer choices, Higher = more diverse choices.
# do_sample: Chooses the decoding method. False = greedy (deterministic), True = sampling (random).
# pad_token_id: Specifies the padding token for shorter sequences, mainly used for batching and to avoid warnings.

In [ ]:
def generate_text(
    prompt,
    temperature=1.0,
    top_p=1.0,
    do_sample=True,
    max_new_tokens=50
):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True) , outputs.shape)

In [ ]:
generate_text(
    "The future of AI is",
    temperature=0.2
)


The future of AI is uncertain. The future of AI is uncertain.

The future of AI is uncertain.

The future of AI is uncertain.

The future of AI is uncertain.

The future of AI is uncertain.

The future of torch.Size([1, 55])


In [ ]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

c:\Users\ASUS\PycharmProjects\Machine-learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 6237.91it/s]


In [ ]:
text = """
Artificial Intelligence has become one of the most influential technologies of the modern era.
It is transforming healthcare, education, transportation, finance, and many other industries.
Researchers continue to develop more capable models while governments and organizations work on regulations to ensure AI is used responsibly.
"""

input_text = "summarize: " + text

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

summary_ids = model.generate(
    **inputs,
    max_new_tokens=60
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

summary

'artificial intelligence is transforming healthcare, education, transportation, finance and many other industries. researchers continue to develop more capable models while governments and organizations work on regulations to ensure AI is used responsibly.'